# CV Masterclass 03: Object Detection Lineage

Welcome to Notebook 03. Object Detection requires the network to answer two questions simultaneously: **What** is it (Classification), and **Where** is it (Localization/Regression).

To understand the dominance of modern YOLO algorithms, we must trace the bloody history of Bounding Box proposals, from the agonizingly slow R-CNN to the elegant speed of Single-Shot Detectors.

---

## 🎓 Socratic Deep Check
Ponder this question as we progress:

> *"A classic YOLO grid detects objects instantly, but completely fails if a hundred tiny birds are clustered together. Why do Anchor Boxes mathematically fix this spatial collision, and why did RetinaNet's 'Focal Loss' obliterate the remaining class imbalances?"*

---

## Table of Contents
1. **The Two-Stage Era:** R-CNN, Fast R-CNN, and Faster R-CNN.
2. **The One-Stage Revolution:** YOLO Grids and Anchor Boxes.
3. **Intersection over Union (IoU) & NMS:** De-duplicating reality.
4. **Focal Loss (RetinaNet):** The mathematical cure for extreme class imbalance.


## 1. The Two-Stage Era (R-CNN Family)

### R-CNN (2014): The Brute Force
R-CNN used a non-neural traditional CV algorithm called "Selective Search" to guess 2,000 possible blobby regions where an object *might* exist in an image.
It then physically cropped those 2,000 regions out, resized them, and passed them sequentially through a ResNet 2,000 separate times. 
**Result:** High accuracy. Incredibly slow (45 seconds per image). You cannot put this in a self-driving car.

### Fast R-CNN (2015): The Shared Map
Instead of cropping 2,000 images, Fast R-CNN passes the *entire* image through the ResNet once to create a massive Feature Map. It then physically projects the 2,000 Selective Search regions directly onto the math of the feature map, extracting them using **RoI Pooling**.
**Result:** 1 GPU pass instead of 2,000. It took 0.3 seconds per image. But Selective Search (running on the CPU) was still the bottleneck.

### Faster R-CNN (2015): The Neural Override
Faster R-CNN deleted Selective Search entirely. It replaced it with a **Region Proposal Network (RPN)**—a tiny secondary neural network that physically slides across the extracted feature map and predicts the bounding boxes natively on the GPU.
**Result:** Real-time object detection was born. However, Two-Stage models (Proposal -> Classification) are still mathematically separated, creating pipeline complexity.

## 2. The One-Stage Revolution (YOLO & SSD)

"You Only Look Once" (YOLO) merged everything into a single mathematical equation.

YOLO takes an image and forcibly overlays a strict $S \times S$ mathematical grid (e.g., $7\times7$ or $13\times13$).

**The YOLO Rule:** If the exact spatial center of an object (e.g., the center pixel of a dog's nose) falls inside Grid Cell 14, then Grid Cell 14 is absolutely uniquely responsible for detecting that dog.

### The Anchor Box Math
What happens if a Car and a Person overlap so perfectly that their center pixels *both* fall into Grid Cell 14? 
Because Grid Cell 14 only outputs one class vector, it is physically impossible to detect both.

**Anchors** solve this by pre-defining 3 physical geometries for every grid cell:
1. Tall and Skinny (To catch standing humans)
2. Short and Wide (To catch cars)
3. Square (To catch boxes)

Now, Grid Cell 14 outputs 3 separate detections. The Human is assigned to the Tall Anchor, the Car to the Wide Anchor. The math survives the collision.

In [ ]:
import torch

# -----------------------------------------------------
# IMPLEMENTATION: Intersection over Union (IoU) & NMS
# -----------------------------------------------------
def calculate_iou(box1, box2):
    """Calculates IoU. Boxes are format [x1, y1, x2, y2]."""
    # 1. Calculate the intersection coordinates
    inter_x1 = max(box1[0], box2[0])
    inter_y1 = max(box1[1], box2[1])
    inter_x2 = min(box1[2], box2[2])
    inter_y2 = min(box1[3], box2[3])

    # 2. Calculate intersecting area (if no overlap, width/height is negative, we clamp to 0)
    inter_width = max(0, inter_x2 - inter_x1)
    inter_height = max(0, inter_y2 - inter_y1)
    intersection_area = inter_width * inter_height

    # 3. Calculate union area
    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union_area = box1_area + box2_area - intersection_area

    return intersection_area / (union_area + 1e-6)


def standard_nms(bboxes, iou_threshold=0.5, confidence_threshold=0.5):
    """
    Non-Maximum Suppression (NMS):
    When YOLO outputs 5 overlapping boxes for 1 dog, NMS destroys the 4 weak ones.
    """
    # 1. Throw away any box that the model is heavily guessing on
    bboxes = [box for box in bboxes if box['score'] > confidence_threshold]
    
    # 2. Sort mathematically by the highest confidence score FIRST
    # (This ensures the "Alpha" box rules over the overlapping "Beta" boxes)
    bboxes = sorted(bboxes, key=lambda x: x['score'], reverse=True)
    
    final_surviving_boxes = []
    
    while bboxes:
        # Grab the highest scoring box remaining
        alpha_box = bboxes.pop(0)
        final_surviving_boxes.append(alpha_box)
        
        # Compare Alpha to all remaining beta boxes. If they overlap severely (>0.5 IoU),
        # they are physically looking at the exact same physical space. Delete them.
        bboxes = [beta for beta in bboxes if calculate_iou(alpha_box['coords'], beta['coords']) < iou_threshold]
        
    return final_surviving_boxes

print("IoU and NMS Deduplication Algorithms defined.")

## 4. RetinaNet & Focal Loss (Curing Class Imbalance)

In One-Stage detectors (YOLO, RetinaNet), a single image might generate $100,000$ Anchor Box predictions.
Of those 100,000 boxes, exactly $10$ might contain an object (a dog or car). The other $99,990$ are purely empty background sky.

Standard Cross-Entropy Loss evaluates every box. Even if the empty sky boxes are "easy" to classify, $99,990 	imes 0.05$ loss creates a massive mathematical avalanche of useless gradients that physically drown out the incredibly important gradients of the $10$ actual objects.

Kaiming He invented **Focal Loss** to fix this. It dynamically scales Cross Entropy based on how confident the model is.

$\text{Focal Loss} = -(1 - P(x))^\gamma \log(P(x))$

If the model is $99\%$ confident a box is empty sky ($P(x) = 0.99$), the $(1 - 0.99)^2$ dynamically multiplies the loss by $0.0001$, instantly muting it. 

If the model is struggling ($P(x) = 0.50$), it leaves the loss high. Focal Loss mathematically forces the neural network to ignore easy background pixels and focus $100\%$ of its computational energy exclusively on "Hard Examples". 